In [28]:
# !pip install fiass

In [29]:
# !pip install openai

In [30]:
# !pip install faiss-cpu

In [31]:
# # requirements: sentence-transformers, faiss-cpu, openai

# from sentence_transformers import SentenceTransformer
# import faiss
# import numpy as np
# import openai

# # Step 1: Prepare documents
# documents = [
#     "Our refund policy: 30 days, full refund with receipt.",
#     "Shipping takes 3-5 business days for domestic orders.",
#     "We accept Visa, Mastercard, and PayPal.",
#     "Customer support: support@example.com or call 1-800-HELP"
# ]

# # Step 2: Create embeddings
# model = SentenceTransformer('all-MiniLM-L6-v2')  # 384-dim embeddings
# embeddings = model.encode(documents)

# # Step 3: Build FAISS index
# dimension = embeddings.shape[1]
# index = faiss.IndexFlatL2(dimension)
# index.add(np.array(embeddings))

# # Step 4: Retrieval function
# def retrieve(query, k=2):
#     query_embedding = model.encode([query])
#     distances, indices = index.search(query_embedding, k)
#     return [documents[i] for i in indices[0]]

# # Step 5: RAG function
# def rag_query(question):
#     # Retrieve relevant docs
#     context = retrieve(question)

#     # Create prompt
#     prompt = f"""Answer the question based only on this context:

# Context:
# {chr(10).join(context)}

# Question: {question}

# Answer:"""

#     # Generate response
#     response = openai.ChatCompletion.create(
#         model="gpt-3.5-turbo",
#         messages=[{"role": "user", "content": prompt}]
#     )

#     return response.choices[0].message.content

# # Test it
# print(rag_query("How long does shipping take?"))
# # Output: "Shipping takes 3-5 business days for domestic orders."

In [32]:
# !pip install -q faiss-cpu sentence-transformers openai

In [33]:
# !pip install -q faiss-cpu sentence-transformers transformers accelerate

In [34]:
# from sentence_transformers import SentenceTransformer
# from transformers import pipeline
# import faiss
# import numpy as np
# import torch

# # -----------------------------
# # Step 1: Prepare documents
# # -----------------------------
# documents = [
#     {
#         "content": "Our refund policy: 30 days, full refund with receipt.",
#         "source": "policy_doc_1",
#         "section": "Refund Policy"
#     },
#     {
#         "content": "Shipping takes 3-5 business days for domestic orders.",
#         "source": "policy_doc_1",
#         "section": "Shipping Policy"
#     },
#     {
#         "content": "We accept Visa, Mastercard, and PayPal.",
#         "source": "payment_doc_1",
#         "section": "Payment Methods"
#     },
#     {
#         "content": "Customer support: support@example.com or call 1-800-HELP.",
#         "source": "support_doc_1",
#         "section": "Customer Support"
#     }
# ]

# texts = [doc["content"] for doc in documents]

# # -----------------------------
# # Step 2: Create embeddings locally
# # -----------------------------
# embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# embeddings = embedding_model.encode(
#     texts,
#     convert_to_numpy=True,
#     normalize_embeddings=True
# ).astype("float32")

# # -----------------------------
# # Step 3: Build FAISS index
# # -----------------------------
# dimension = embeddings.shape[1]

# # With normalized embeddings, inner product works like cosine similarity.
# # FAISS documentation notes cosine search can be done by normalizing vectors first.
# index = faiss.IndexFlatIP(dimension)
# index.add(embeddings)

# # -----------------------------
# # Step 4: Retrieval function
# # -----------------------------
# def retrieve(query, k=2):
#     query_embedding = embedding_model.encode(
#         [query],
#         convert_to_numpy=True,
#         normalize_embeddings=True
#     ).astype("float32")

#     scores, indices = index.search(query_embedding, k)

#     retrieved_docs = []
#     for score, idx in zip(scores[0], indices[0]):
#         retrieved_docs.append({
#             "content": documents[idx]["content"],
#             "source": documents[idx]["source"],
#             "section": documents[idx]["section"],
#             "score": float(score)
#         })

#     return retrieved_docs

# # -----------------------------
# # Step 5: Load local Hugging Face model
# # -----------------------------
# device = 0 if torch.cuda.is_available() else -1

# generator = pipeline(
#     task="text2text-generation",
#     model="google/flan-t5-base",
#     device=device
# )

# # -----------------------------
# # Step 6: RAG function with local generation
# # -----------------------------
# def rag_query(question, k=2):
#     retrieved_docs = retrieve(question, k=k)

#     context = "\n".join(
#         [
#             f"Source: {doc['source']} | Section: {doc['section']} | Content: {doc['content']}"
#             for doc in retrieved_docs
#         ]
#     )

#     prompt = f"""
# Answer the question using only the context.

# Context:
# {context}

# Question:
# {question}

# Rules:
# - If the answer is in the context, answer directly.
# - Mention the source and section.
# - If the answer is not in the context, say: I do not have enough information in the provided context.

# Answer:
# """

#     output = generator(
#         prompt,
#         max_new_tokens=120,
#         do_sample=False
#     )

#     answer = output[0]["generated_text"]

#     return {
#         "question": question,
#         "answer": answer,
#         "retrieved_docs": retrieved_docs
#     }

# # -----------------------------
# # Step 7: Test
# # -----------------------------
# result = rag_query("How long does shipping take?", k=2)

# print("Answer:")
# print(result["answer"])

# print("\nRetrieved Documents:")
# for doc in result["retrieved_docs"]:
#     print(f"- {doc['source']} | {doc['section']} | score={doc['score']:.4f}")
#     print(f"  {doc['content']}")

In [35]:
# from sentence_transformers import SentenceTransformer
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# import faiss
# import numpy as np
# import torch

# # -----------------------------
# # Step 1: Prepare documents
# # -----------------------------
# documents = [
#     {
#         "content": "Our refund policy: 30 days, full refund with receipt.",
#         "source": "policy_doc_1",
#         "section": "Refund Policy"
#     },
#     {
#         "content": "Shipping takes 3-5 business days for domestic orders.",
#         "source": "policy_doc_1",
#         "section": "Shipping Policy"
#     },
#     {
#         "content": "We accept Visa, Mastercard, and PayPal.",
#         "source": "payment_doc_1",
#         "section": "Payment Methods"
#     },
#     {
#         "content": "Customer support: support@example.com or call 1-800-HELP.",
#         "source": "support_doc_1",
#         "section": "Customer Support"
#     }
# ]

# texts = [doc["content"] for doc in documents]

# # -----------------------------
# # Step 2: Create embeddings locally
# # -----------------------------
# embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# embeddings = embedding_model.encode(
#     texts,
#     convert_to_numpy=True,
#     normalize_embeddings=True
# ).astype("float32")

# # -----------------------------
# # Step 3: Build FAISS index
# # -----------------------------
# dimension = embeddings.shape[1]

# index = faiss.IndexFlatIP(dimension)
# index.add(embeddings)

# # -----------------------------
# # Step 4: Retrieval function
# # -----------------------------
# def retrieve(query, k=2):
#     query_embedding = embedding_model.encode(
#         [query],
#         convert_to_numpy=True,
#         normalize_embeddings=True
#     ).astype("float32")

#     scores, indices = index.search(query_embedding, k)

#     retrieved_docs = []
#     for score, idx in zip(scores[0], indices[0]):
#         retrieved_docs.append({
#             "content": documents[idx]["content"],
#             "source": documents[idx]["source"],
#             "section": documents[idx]["section"],
#             "score": float(score)
#         })

#     return retrieved_docs

# # -----------------------------
# # Step 5: Load local Hugging Face model
# # -----------------------------
# model_name = "google/flan-t5-base"

# tokenizer = AutoTokenizer.from_pretrained(model_name)
# generator_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# generator_model = generator_model.to(device)

# # -----------------------------
# # Step 6: RAG function with local generation
# # -----------------------------
# def rag_query(question, k=2):
#     retrieved_docs = retrieve(question, k=k)

#     context = "\n".join(
#         [
#             f"Source: {doc['source']} | Section: {doc['section']} | Content: {doc['content']}"
#             for doc in retrieved_docs
#         ]
#     )

#     prompt = f"""
# Answer the question using only the context.

# Context:
# {context}

# Question:
# {question}

# Rules:
# - If the answer is in the context, answer directly.
# - Mention the source and section.
# - If the answer is not in the context, say: I do not have enough information in the provided context.

# Answer:
# """

#     inputs = tokenizer(
#         prompt,
#         return_tensors="pt",
#         truncation=True,
#         max_length=512
#     ).to(device)

#     outputs = generator_model.generate(
#         **inputs,
#         max_new_tokens=120,
#         do_sample=False
#     )

#     answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

#     return {
#         "question": question,
#         "answer": answer,
#         "retrieved_docs": retrieved_docs
#     }

# # -----------------------------
# # Step 7: Test
# # -----------------------------
# result = rag_query("How long does shipping take?", k=2)

# print("Answer:")
# print(result["answer"])

# print("\nRetrieved Documents:")
# for doc in result["retrieved_docs"]:
#     print(f"- {doc['source']} | {doc['section']} | score={doc['score']:.4f}")
#     print(f"  {doc['content']}")

# build from here

In [1]:
!pip install -q pypdf faiss-cpu sentence-transformers transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.3/346.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 67.6 MB/s eta 0:00:00


In [2]:
from google.colab import files
import os
import shutil

upload_folder = "pdf_files"
os.makedirs(upload_folder, exist_ok=True)

uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, os.path.join(upload_folder, filename))

print("Uploaded PDF files:")
print(os.listdir(upload_folder))

Saving SCC_CFP_2025-07-Main-Document_v.1_2025-09-05_EN.pdf to SCC_CFP_2025-07-Main-Document_v.1_2025-09-05_EN.pdf
Saving RFP CP-730126 Generative AI RFP.pdf to RFP CP-730126 Generative AI RFP.pdf
Uploaded PDF files:
['SCC_CFP_2025-07-Main-Document_v.1_2025-09-05_EN.pdf', 'RFP CP-730126 Generative AI RFP.pdf']


In [3]:
from pypdf import PdfReader
import os

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    pages_data = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text()

        if text and text.strip():
            pages_data.append({
                "source": os.path.basename(pdf_path),
                "page": page_number,
                "text": text.strip()
            })

    return pages_data


all_pages = []

for filename in os.listdir(upload_folder):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(upload_folder, filename)
        pages = extract_text_from_pdf(pdf_path)
        all_pages.extend(pages)

print(f"Total extracted pages: {len(all_pages)}")

for item in all_pages[:2]:
    print("Source:", item["source"])
    print("Page:", item["page"])
    print("Text preview:", item["text"][:300])
    print("-" * 80)

Total extracted pages: 77
Source: SCC_CFP_2025-07-Main-Document_v.1_2025-09-05_EN.pdf
Page: 1
Text preview: 1 
 
 
 
Call for Proposals (CFP-2025-07)  
Advancing Artificial Intelligence 
through Standardization Strategies and 
Tools  
Issued: September 5, 2025 
Bidder questions deadline: September 22, 2025 at 2:00 PM ET 
Submission deadline: October 6, 2025 at 2:00 PM ET 
Tender contact: Martin Larocque 

--------------------------------------------------------------------------------
Source: SCC_CFP_2025-07-Main-Document_v.1_2025-09-05_EN.pdf
Page: 2
Text preview: 1 
TABLE OF CONTENTS 
1. Overview ....................................................................................................................... 3 
1.1 SCC’s Artificial Intelligence and Data Governance Standardization program ... 3 
1.2 Objective of the Call for Proposals...................
--------------------------------------------------------------------------------


In [4]:
def chunk_text(text, chunk_size=1000, overlap=120):
    chunks = []

    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk.strip())

        start = end - overlap

    return chunks


documents = []
chunk_id = 0

for page in all_pages:
    chunks = chunk_text(
        page["text"],
        chunk_size=700,
        overlap=120
    )

    for chunk in chunks:
        documents.append({
            "doc_id": f"chunk_{chunk_id}",
            "content": chunk,
            "source": page["source"],
            "page": page["page"],
            "tags": []
        })
        chunk_id += 1

print(f"Total chunks: {len(documents)}")

for doc in documents[:2]:
    print(doc["doc_id"], doc["source"], "page:", doc["page"])
    print(doc["content"][:300])
    print("-" * 80)

Total chunks: 335
chunk_0 SCC_CFP_2025-07-Main-Document_v.1_2025-09-05_EN.pdf page: 1
1 
 
 
 
Call for Proposals (CFP-2025-07)  
Advancing Artificial Intelligence 
through Standardization Strategies and 
Tools  
Issued: September 5, 2025 
Bidder questions deadline: September 22, 2025 at 2:00 PM ET 
Submission deadline: October 6, 2025 at 2:00 PM ET 
Tender contact: Martin Larocque 

--------------------------------------------------------------------------------
chunk_1 SCC_CFP_2025-07-Main-Document_v.1_2025-09-05_EN.pdf page: 2
1 
TABLE OF CONTENTS 
1. Overview ....................................................................................................................... 3 
1.1 SCC’s Artificial Intelligence and Data Governance Standardization program ... 3 
1.2 Objective of the Call for Proposals...................
--------------------------------------------------------------------------------


In [5]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [doc["content"] for doc in documents]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("Number of vectors stored in FAISS:", index.ntotal)
print("Embedding dimension:", dimension)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Number of vectors stored in FAISS: 335
Embedding dimension: 384


In [6]:
import json
import os
import faiss

storage_folder = "rag_storage"
os.makedirs(storage_folder, exist_ok=True)

faiss.write_index(index, os.path.join(storage_folder, "rfp_index.faiss"))

with open(os.path.join(storage_folder, "rfp_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print("Saved:")
print("- rag_storage/rfp_index.faiss")
print("- rag_storage/rfp_metadata.json")

Saved:
- rag_storage/rfp_index.faiss
- rag_storage/rfp_metadata.json


In [7]:
import json
import faiss
import os

storage_folder = "rag_storage"

index = faiss.read_index(os.path.join(storage_folder, "rfp_index.faiss"))

with open(os.path.join(storage_folder, "rfp_metadata.json"), "r", encoding="utf-8") as f:
    documents = json.load(f)

print("Loaded FAISS vectors:", index.ntotal)
print("Loaded metadata records:", len(documents))

Loaded FAISS vectors: 335
Loaded metadata records: 335


In [8]:
def retrieve(query, k=5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        doc = documents[idx]

        results.append({
            "content": doc["content"],
            "source": doc["source"],
            "page": doc["page"],
            "doc_id": doc["doc_id"],
            "score": float(score)
        })

    return results


results = retrieve("What experience do we have in AI training?", k=6)

for result in results:
    print(f"Source: {result['source']} | Page: {result['page']} | Score: {result['score']:.4f}")
    print(result["content"][:1000])
    print("-" * 80)

Source: RFP CP-730126 Generative AI RFP.pdf | Page: 24 | Score: 0.5852
ve successfully handled requests from 
Higher Educational Institutions? 
  
 What is your team's background in generative AI and 
related fields? 
  
Features   What are the key features of your generative AI 
software? 
  
 How quickly does the software generate outputs? Is it 
instantaneous? 
  
 Can you speak to the usability, including the software's 
user interface and user experience? 
  
Training   Types of training offered (i.e. self-learning, instructor led, 
one on one, train the trainer). 
  
 Method of delivery.
--------------------------------------------------------------------------------
Source: RFP CP-730126 Generative AI RFP.pdf | Page: 25 | Score: 0.5075
RFP No. CP-730126   Generative Artificial Intelligence (AI) Software  Page 25 of 33 
 
 Number of hours of training included in the pricing 
structure. 
  
 Number of people included for training in the pricing 
structure. 
  
 Recommenda

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
generator_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator_model = generator_model.to(device)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [10]:
def rag_query(question, k=5):
    retrieved_docs = retrieve(question, k=k)

    context = "\n\n".join(
        [
            f"Source: {doc['source']} | Page: {doc['page']} | Chunk ID: {doc['doc_id']}\nContent: {doc['content']}"
            for doc in retrieved_docs
        ]
    )

    prompt = f"""
You are an RFP assistant. Use ONLY the provided context to answer.

Context:
{context}

Question:
{question}

Instructions:
- Answer using only the context.
- If the answer is not in the context, say: I do not have enough information in the provided context.
- Cite the source PDF and page number.
- Keep the answer clear and professional.

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    outputs = generator_model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return {
        "question": question,
        "answer": answer,
        "sources": [
            {
                "source": doc["source"],
                "page": doc["page"],
                "doc_id": doc["doc_id"],
                "score": doc["score"]
            }
            for doc in retrieved_docs
        ],
        "retrieved_docs": retrieved_docs
    }


result = rag_query("What experience do we have in AI training?", k=5)

print("Answer:")
print(result["answer"])

print("\nSources used:")
for source in result["sources"]:
    print(f"- {source['source']} | page {source['page']} | {source['doc_id']} | score={source['score']:.4f}")

Answer:
Answer using only the context.

Sources used:
- RFP CP-730126 Generative AI RFP.pdf | page 24 | chunk_304 | score=0.5852
- RFP CP-730126 Generative AI RFP.pdf | page 25 | chunk_305 | score=0.5075
- RFP CP-730126 Generative AI RFP.pdf | page 22 | chunk_293 | score=0.4888
- SCC_CFP_2025-07-Main-Document_v.1_2025-09-05_EN.pdf | page 22 | chunk_90 | score=0.4375
- RFP CP-730126 Generative AI RFP.pdf | page 24 | chunk_303 | score=0.4227


In [11]:
def rag_query(question, k=5):
    retrieved_docs = retrieve(question, k=k)

    context = "\n\n".join(
        [
            f"Source: {doc['source']} | Page: {doc['page']} | Chunk ID: {doc['doc_id']}\nContent: {doc['content']}"
            for doc in retrieved_docs
        ]
    )

    prompt = f"""
You are an RFP assistant. Use ONLY the provided context to answer.

Context:
{context}

Question:
{question}

Instructions:
- Answer using only the context.
- If the answer is not in the context, say: I do not have enough information in the provided context.
- Cite the source PDF and page number.
- Keep the answer clear and professional.

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    outputs = generator_model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return {
        "question": question,
        "answer": answer,
        "sources": [
            {
                "source": doc["source"],
                "page": doc["page"],
                "doc_id": doc["doc_id"],
                "score": doc["score"]
            }
            for doc in retrieved_docs
        ],
        "retrieved_docs": retrieved_docs
    }


result = rag_query("What experience do we have in AI training?", k=5)

print("Answer:")
print(result["answer"])

print("\nSources used:")
for source in result["sources"]:
    print(f"- {source['source']} | page {source['page']} | {source['doc_id']} | score={source['score']:.4f}")

Answer:
Answer using only the context.

Sources used:
- RFP CP-730126 Generative AI RFP.pdf | page 24 | chunk_304 | score=0.5852
- RFP CP-730126 Generative AI RFP.pdf | page 25 | chunk_305 | score=0.5075
- RFP CP-730126 Generative AI RFP.pdf | page 22 | chunk_293 | score=0.4888
- SCC_CFP_2025-07-Main-Document_v.1_2025-09-05_EN.pdf | page 22 | chunk_90 | score=0.4375
- RFP CP-730126 Generative AI RFP.pdf | page 24 | chunk_303 | score=0.4227
